In [1]:
!pip install matplotlib numpy ipywidgets

In [26]:
# Kód pro bresenhamův algoritmus

def bresenham_didactic(x0, y0, x1, y1):
    """
    Bresenhamův algoritmus pro 1. oktant (dx >= dy >= 0) se záznamem kroků.
    Vrací seznam slovníků, kde každý slovník popisuje jeden krok rasterizace.
    """
    dx = x1 - x0
    dy = y1 - y0

    # Počáteční hodnota rozhodovací proměnné (prediktoru)
    P = 2 * dy - dx

    x, y = x0, y0
    steps = []
    pixels = []

    for k in range(dx + 1):
        pixels.append((x, y))
        P_current = P

        # Rozhodovací logika pro další pixel
        if P < 0:
            rozhodnuti = f"P ({P_current}) < 0 → Y se nemění ({y}), P_nové = {P_current} + 2*dy ({2*dy}) = {P_current + 2*dy}"
            P = P + 2 * dy
            next_y = y
        else:
            rozhodnuti = f"P ({P_current}) >= 0 → Y roste o 1 ({y + 1}), P_nové = {P_current} + 2*dy - 2*dx ({2*dy - 2*dx}) = {P_current + 2 * dy - 2 * dx}"
            P = P + 2 * dy - 2 * dx
            next_y = y + 1

        steps.append({
            "krok": k,
            "aktualni_pixel": (x, y),
            "vykreslene_pixely": list(pixels),
            "P_pred": P_current,
            "P_po": P,
            "popis": rozhodnuti
        })

        x += 1
        y = next_y

    return steps

In [25]:
# Kód pro vykreslení algoritmu do grafu

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display

# Pevně dané souřadnice a předpočítání kroků
X0, Y0 = 1, 1
X1, Y1 = 11, 8
historie = bresenham_didactic(X0, Y0, X1, Y1)
max_krok = len(historie) - 1

aktualni_krok = 0

# Vytvoření ovládacích prvků
btn_zpet = widgets.Button(description='<- Krok zpět', disabled=True, button_style='info')
btn_vpred = widgets.Button(description='Krok vpřed ->', disabled=False, button_style='info')
lbl_stav = widgets.Label(value=f"Krok: 0 / {max_krok}")
vystup_grafu = widgets.Output(layout=widgets.Layout(min_height='620px'))

def vykresli_aktualni_stav():
    data_kroku = historie[aktualni_krok]
    cur_x, cur_y = data_kroku["aktualni_pixel"]
    P_pred = data_kroku["P_pred"]
    P_po = data_kroku["P_po"]

    max_x = X1 + 3
    max_y = Y1 + 3

    # Popis nad graf
    if aktualni_krok == max_krok:
        nadpis = (
            f"Krok {aktualni_krok} z {max_krok} | Jsme v cílovém pixelu [{cur_x}, {cur_y}]\n"
            f"Úsečka je kompletně vykreslená."
        )
    else:
        if P_pred < 0:
            vysvetleni = (
                f"Chyba P je záporná ({P_pred} < 0) → Ideální čára vede spíše dolu.\n"
                f"Rozhodnutí: Jdeme ROVNĚ DOPRAVA na pixel [{cur_x + 1}, {cur_y}] (výška Y zůstává {cur_y}). "
                f"Nový prediktor P = {P_po}."
            )
        else:
            vysvetleni = (
                f"Chyba P je kladná nebo nula ({P_pred} ≥ 0) → Ideální čára vede spíše nahoru.\n"
                f"Rozhodnutí: Jdeme ŠIKMO NAHORU na pixel [{cur_x + 1}, {cur_y + 1}] (výška Y stoupne na {cur_y + 1}). "
                f"Nový prediktor P = {P_po}."
            )
        nadpis = f"Krok {aktualni_krok} z {max_krok} | Stojíme na pixelu [{cur_x}, {cur_y}]\n" + vysvetleni

    with vystup_grafu:
        vystup_grafu.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(9, 6))

        # Vykreslení již rasterizovaných pixelů
        for px, py in data_kroku["vykreslene_pixely"]:
            je_aktualni = (px, py) == (cur_x, cur_y)
            barva = '#ff6b6b' if je_aktualni else '#4ecdc4'
            alfa = 0.9 if je_aktualni else 0.5

            rect = patches.Rectangle((px - 0.5, py - 0.5), 1, 1, 
                                     linewidth=1.5, edgecolor='black', 
                                     facecolor=barva, alpha=alfa, zorder=2)
            ax.add_patch(rect)

        # Ideální matematická úsečka
        ax.plot([X0, X1], [Y0, Y1], color='#1a535c', linestyle='--', linewidth=2, label='Ideální úsečka', zorder=3)
        ax.scatter([X0, X1], [Y0, Y1], color='#1a535c', s=60, zorder=4)

        # Střed aktuálního pixelu a kandidáti na další krok
        ax.scatter([cur_x], [cur_y], color='red', s=40, zorder=5, label='Střed aktuálního pixelu')

        if aktualni_krok < max_krok:
            ax.scatter([cur_x + 1, cur_x + 1], [cur_y, cur_y + 1], 
                       color='orange', edgecolor='black', s=80, zorder=5, label='Kandidáti na další vykreslení')

        # Nastavení mřížky
        ax.set_xticks(range(0, max_x))
        ax.set_yticks(range(0, max_y))
        ax.set_xticks([x - 0.5 for x in range(0, max_x + 1)], minor=True)
        ax.set_yticks([y - 0.5 for y in range(0, max_y + 1)], minor=True)

        ax.grid(which='minor', color='#cccccc', linestyle='-', linewidth=1)
        ax.grid(which='major', color='#eeeeee', linestyle=':', linewidth=0.5)

        ax.set_xlim(-0.5, max_x - 0.5)
        ax.set_ylim(-0.5, max_y - 0.5)
        ax.set_aspect('equal')

        # Panel nad grafem
        ax.set_title(
            nadpis, 
            fontsize=10.5, pad=14, loc='left',
            bbox=dict(boxstyle='round,pad=0.6', facecolor='#f8f9fa', edgecolor='#dee2e6')
        )

        ax.legend(loc='upper left')
        plt.tight_layout()
        plt.show()

# Obsluha kliknutí na tlačítka
def pri_kliknuti_zpet(b):
    global aktualni_krok
    if aktualni_krok > 0:
        aktualni_krok -= 1
        aktualizuj_ui()

def pri_kliknuti_vpred(b):
    global aktualni_krok
    if aktualni_krok < max_krok:
        aktualni_krok += 1
        aktualizuj_ui()

def aktualizuj_ui():
    btn_zpet.disabled = (aktualni_krok == 0)
    btn_vpred.disabled = (aktualni_krok == max_krok)
    lbl_stav.value = f"Krok: {aktualni_krok} / {max_krok}"
    vykresli_aktualni_stav()

btn_zpet.on_click(pri_kliknuti_zpet)
btn_vpred.on_click(pri_kliknuti_vpred)

# Složení panelu a vykreslení
ovladaci_panel = widgets.HBox([btn_zpet, btn_vpred, lbl_stav])
#display(vystup_grafu, ovladaci_panel)
#vykresli_aktualni_stav()

## Princip a pseudokód Bresenhamova algoritmu

Bresenhamův algoritmus vykresluje úsečku pomocí **celočíselné aritmetiky**. V každém kroku se posune o 1 pixel doprava v ose $X$ a pomocí znaménka rozhodovací proměnné (prediktoru $P$) určí, zda má souřadnice $Y$ zůstat stejná, nebo se zvýšit o 1.

```text
ALGORITMUS: Bresenhamův algoritmus
VSTUP:  Počáteční bod [x0, y0], koncový bod [x1, y1]
VÝSTUP: Vykreslené pixely úsečky

1. Spočítej vzdálenosti v osách:
   dx = x1 - x0
   dy = y1 - y0

2. Nastav počáteční hodnotu prediktoru a výchozí výšku:
   P = 2 * dy - dx
   y = y0

3. Pro každé x od x0 do x1 opakuj:
   a) Vykresli_Pixel(x, y)
   
   b) Rozhodni o výšce Y pro následující krok:
      POKUD je P < 0 PAK:
          // Ideální přímka vede spíše rovně -> Y se nemění
          P = P + 2 * dy
      JINAK (tedy P >= 0):
          // Ideální přímka vede spíše nahoru -> Y stoupne o 1
          y = y + 1
          P = P + 2 * dy - 2 * dx
```

In [27]:
display(vystup_grafu, ovladaci_panel)
vykresli_aktualni_stav()

Output(layout=Layout(min_height='620px'))